Data Cleaning


In [1]:
import json
import pandas as pd

with open("data_products.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data["products"])

In [2]:
df["description"] = (
    df["description"]
    .fillna("")
    .str.replace("Ko'proq ko'rsatish", "", regex=False)
    .str.strip()
)

In [3]:
df["price_text"] = df["price.text"]
df["price_amount"] = df["price.amount"]
df["price_currency"] = df["price.currency"]

In [4]:
def get_category(breadcrumbs):
    if not isinstance(breadcrumbs, list):
        return None

    names = [item.get("name") for item in breadcrumbs if item.get("name")]

    if len(names) >= 3:
        return names[-2]

    if len(names) >= 2 and names[0] != "Bosh sahifa":
        return names[-1]

    return None

df["category"] = df["breadcrumbs"].apply(get_category)

In [5]:
def make_content(row):
    parts = [
        f"Mahsulot: {row.get('name')}",
        f"Kategoriya: {row.get('category')}",
        f"Narxi: {row.get('price_text')}",
        f"Yetkazib berish: {row.get('delivery')}",
        f"Kafolat: {row.get('warranty')}",
        f"Tavsif: {row.get('description')}",
    ]

    return "\n".join([p for p in parts if p and "nan" not in p.lower()])

df["content"] = df.apply(make_content, axis=1)

In [6]:
df_chunks = df[[
    "id",
    "name",
    "source_url",
    "category",
    "price_text",
    "price_amount",
    "price_currency",
    "content"
]].copy()

df_chunks.to_json(
    "product-chunks-pandas.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)